# Cryptocurrency Data Scraper - Bitget Exchange

This notebook scrapes cryptocurrency data from Bitget exchange using the Pydoll library with concurrent processing for improved performance.

## Features:
- ✅ Concurrent scraping of multiple cryptocurrencies
- ✅ Retry logic for failed requests
- ✅ Excel export functionality
- ✅ Error handling and statistics
- ✅ Real-time progress tracking

## 1. Import Required Libraries

Import all necessary libraries for web scraping, data processing, and async operations.

In [1]:
import asyncio
import pandas as pd
from pydoll.browser.chrome import Chrome
from pydoll.constants import By
import time
from datetime import datetime

print("📚 All libraries imported successfully!")
print(f"🕐 Notebook started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

📚 All libraries imported successfully!
🕐 Notebook started at: 2025-06-04 15:31:54


## 2. Configuration & Settings

Define the list of cryptocurrencies to scrape and other configuration parameters.

In [2]:
# List of cryptocurrencies to scrape
list_coin = [
    'bitcoin', 
    'ethereum', 
    'binance', 
    'solana'
]

# Configuration parameters
MAX_RETRIES = 2
WAIT_TIME = 3  # seconds to wait for page load
RETRY_DELAY = 2  # seconds between retries
ELEMENT_TIMEOUT = 10  # seconds to wait for elements

print(f"🎯 Target cryptocurrencies: {', '.join(list_coin)}")
print(f"⚙️ Max retries: {MAX_RETRIES}, Wait time: {WAIT_TIME}s, Element timeout: {ELEMENT_TIMEOUT}s")

🎯 Target cryptocurrencies: bitcoin, ethereum, binance, solana
⚙️ Max retries: 2, Wait time: 3s, Element timeout: 10s


## 3. Helper Functions

Define utility functions for text extraction using Chrome DevTools Protocol (CDP).

In [3]:
async def get_element_text(page, element):
    """Helper function to get text from element using CDP commands"""
    try:
        if element and hasattr(element, '_object_id'):
            # Use Runtime.callFunctionOn to get text content
            command = {
                "id": 1,
                "method": "Runtime.callFunctionOn",
                "params": {
                    "functionDeclaration": "function() { return this.textContent || this.innerText || ''; }",
                    "objectId": element._object_id,
                    "returnByValue": True
                }
            }
            
            result = await page._connection_handler.execute_command(command)
            
            if result and 'result' in result and 'result' in result['result'] and 'value' in result['result']['result']:
                text = result['result']['result']['value'].strip()
                return text if text else "N/A"
            else:
                return "N/A"
        else:
            return "N/A"
    except Exception as e:
        print(f"⚠️ Error getting text from element: {e}")
        return "N/A"

print("✅ Helper functions defined successfully!")

✅ Helper functions defined successfully!


## 4. Main Scraping Function

Core function to fetch data from a single cryptocurrency page.

In [4]:
async def fetch_coin_data(coin):
    """Fetch cryptocurrency data from Bitget for a single coin"""
    url = f'https://www.bitget.com/price/{coin}'
    print(f"🔎 Starting crawl: {coin}")

    browser = Chrome()
    
    try:
        await browser.start()
        page = await browser.get_page()
        
        await page.go_to(url)
        print(f"📄 Page loaded: {coin}")
        
        await asyncio.sleep(WAIT_TIME)

        try:
            # Get price using exact XPath (same as Selenium)
            price_elem = await page.wait_element(
                By.XPATH, 
                '//span[@class="font-bold text-[40px] ltIpad:text-[32px] leading-[48px] ltIpad:leading-[38px] text-primaryText"]',
                timeout=ELEMENT_TIMEOUT
            )
            price = await get_element_text(page, price_elem) if price_elem else "N/A"

            # Get date using exact XPath
            try:
                date_elem = await page.wait_element(
                    By.XPATH, 
                    '//div[@class="text-[14px] mt-[24px] text-thirdText font-medium"]',
                    timeout=5
                )
                date = await get_element_text(page, date_elem) if date_elem else "N/A"
            except Exception:
                date = "N/A"

            data = {'coin': coin, 'price': price, 'date': date}
            
            try:
                # Get additional metrics using exact XPath
                labels = await page.find_elements(
                    By.XPATH, 
                    '//span[@class="text-[14px] text-[var(--content-secondary)]"]'
                )
                values = await page.find_elements(
                    By.XPATH, 
                    '//span[@class="text-[14px] font-[600]"]'
                )

                if labels and values:
                    keys = [await get_element_text(page, el) for el in labels[:5]]
                    vals = [await get_element_text(page, el) for el in values[:5]]
                    
                    limit = min(len(keys), len(vals))
                    for i in range(limit):
                        if keys[i] and vals[i] and keys[i] != "N/A" and vals[i] != "N/A":
                            clean_key = keys[i].strip(':').strip()
                            clean_val = vals[i].strip()
                            if clean_key and clean_val:
                                data[clean_key] = clean_val

            except Exception as e:
                print(f"⚠️ Unable to fetch additional info for {coin}: {e}")

            print(f"✅ Crawl completed: {coin}")
            return data

        except Exception as e:
            print(f"❌ Error finding element for {coin}: {e}")
            return {'coin': coin, 'error': str(e)}

    except Exception as e:
        print(f"❌ Error crawling {coin}: {e}")
        return {'coin': coin, 'error': str(e)}

    finally:
        try:
            await browser.stop()
        except Exception as e:
            print(f"⚠️ Error closing browser for {coin}: {e}")

print("✅ Main scraping function defined successfully!")

✅ Main scraping function defined successfully!


## 5. Retry Logic Function

Function with retry mechanism for handling failed requests.

In [5]:
async def crawl_single_coin(coin):
    """Crawl a single coin with retry logic"""
    for attempt in range(MAX_RETRIES):
        try:
            result = await fetch_coin_data(coin)
            if 'error' not in result:
                return result
            else:
                print(f"🔄 Retrying {coin} attempt {attempt + 1}")
                await asyncio.sleep(RETRY_DELAY)
        except Exception as e:
            print(f"🔄 Error attempt {attempt + 1} for {coin}: {e}")
            if attempt < MAX_RETRIES - 1:
                await asyncio.sleep(3)
    
    return {'coin': coin, 'error': 'Failed after retries'}

print("✅ Retry logic function defined successfully!")

✅ Retry logic function defined successfully!


## 6. Concurrent Processing Function

Main orchestration function that handles concurrent scraping of all cryptocurrencies.

In [6]:
async def scrape_all_coins():
    """Main function to scrape all cryptocurrencies concurrently"""
    print("🚀 Starting cryptocurrency data crawling...")
    print(f"🧪 Testing with {len(list_coin)} coins concurrently...")
    
    try:
        # Create tasks for all coins simultaneously
        tasks = []
        for coin in list_coin:
            task = asyncio.create_task(crawl_single_coin(coin))
            tasks.append(task)
            print(f"📋 Created task for {coin}")
        
        print(f"🚀 Starting {len(tasks)} concurrent crawl tasks...")
        
        # Run all tasks simultaneously and wait for results
        all_data = await asyncio.gather(*tasks, return_exceptions=True)
        
        # Process results and exceptions
        processed_data = []
        for i, result in enumerate(all_data):
            coin = list_coin[i]
            if isinstance(result, Exception):
                print(f"❌ Exception for {coin}: {result}")
                processed_data.append({'coin': coin, 'error': str(result)})
            else:
                processed_data.append(result)
        
        print(f"📋 All crawl tasks completed")
        return processed_data
    
    except Exception as e:
        print(f"❌ Critical error in scraping: {e}")
        import traceback
        traceback.print_exc()
        return []

print("✅ Concurrent processing function defined successfully!")

✅ Concurrent processing function defined successfully!


## 7. Data Export Function

Function to save scraped data to Excel file with statistics.

In [7]:
def save_to_excel(processed_data, filename="bitget_coin_data.xlsx"):
    """Save processed data to Excel file with statistics"""
    try:
        df = pd.DataFrame(processed_data)
        df.to_excel(filename, index=False)
        print(f"📄 Data saved to {filename}")
        
        # Calculate statistics
        success_count = len([d for d in processed_data if 'error' not in d])
        error_count = len(processed_data) - success_count
        
        print(f"📊 Statistics:")
        print(f"   ✅ Successes: {success_count}")
        print(f"   ❌ Errors: {error_count}")
        print(f"   📈 Success Rate: {(success_count/len(processed_data)*100):.1f}%")
        
        return df
        
    except Exception as e:
        print(f"❌ Error saving Excel file: {e}")
        return None

print("✅ Data export function defined successfully!")

✅ Data export function defined successfully!


## 8. Execute Scraping Process

**Run this cell to start the scraping process!**

This will scrape all configured cryptocurrencies concurrently and display progress in real-time.

In [8]:
# Main execution cell
print("🎬 Starting scraping process...")
start_time = time.time()

try:
    # Run the scraping process
    scraped_data = await scrape_all_coins()
    
    if scraped_data:
        # Save to Excel
        df_result = save_to_excel(scraped_data)
        
        # Display first few rows
        if df_result is not None:
            print("\n📋 First few rows of scraped data:")
            display(df_result.head())
    else:
        print("❌ No data was scraped successfully")
    
    end_time = time.time()
    print(f"\n⏱️ Total execution time: {end_time - start_time:.2f} seconds")
    
except Exception as e:
    print(f"❌ Critical error in execution: {e}")
    import traceback
    traceback.print_exc()

🎬 Starting scraping process...
🚀 Starting cryptocurrency data crawling...
🧪 Testing with 4 coins concurrently...
📋 Created task for bitcoin
📋 Created task for ethereum
📋 Created task for binance
📋 Created task for solana
🚀 Starting 4 concurrent crawl tasks...
🔎 Starting crawl: bitcoin
🔎 Starting crawl: ethereum
🔎 Starting crawl: binance
🔎 Starting crawl: solana
📄 Page loaded: binance
📄 Page loaded: binance
📄 Page loaded: solana
📄 Page loaded: solana
✅ Crawl completed: binance
✅ Crawl completed: binance
✅ Crawl completed: solana
✅ Crawl completed: solana
📄 Page loaded: bitcoin
📄 Page loaded: bitcoin
📄 Page loaded: ethereum
📄 Page loaded: ethereum
✅ Crawl completed: bitcoin
✅ Crawl completed: bitcoin
✅ Crawl completed: ethereum
✅ Crawl completed: ethereum
📋 All crawl tasks completed
📋 All crawl tasks completed
📄 Data saved to bitget_coin_data.xlsx
📊 Statistics:
   ✅ Successes: 4
   ❌ Errors: 0
   📈 Success Rate: 100.0%

📋 First few rows of scraped data:
📄 Data saved to bitget_coin_data.x

,coin,price,date,Market cap,Fully diluted market cap,Volume (24h),24h volume / market cap,24h high
0,bitcoin,"$105,624.65",Last updated as of 2025-06-04 08:31:52（UTC+0）,"$2,099,205,436,431.22","$2,099,205,436,431.22","$44,509,268,141.79",2.12%,"$106,863.55"
1,ethereum,"$2,637.98",Last updated as of 2025-06-04 08:28:35（UTC+0）,"$318,464,212,981.91","$318,464,212,981.91","$16,440,607,837.49",5.16%,"$2,652.42"
2,binance,$671.57,Last updated as of 2025-06-04 08:29:56（UTC+0）,"$94,616,160,169.44","$94,616,160,169.44","$1,637,858,051.07",1.73%,$671.73
3,solana,$156.85,Last updated as of 2025-06-04 08:28:45（UTC+0）,"$81,961,487,082.61","$81,961,487,082.61","$3,336,282,104.86",4.07%,$163.47



⏱️ Total execution time: 53.86 seconds


## 9. Data Analysis & Visualization

Analyze the scraped data and create visualizations.

In [9]:
# Data analysis cell
try:
    if 'df_result' in locals() and df_result is not None:
        print("📊 Data Analysis:")
        print(f"\n🔢 Dataset Info:")
        print(f"   • Total rows: {len(df_result)}")
        print(f"   • Total columns: {len(df_result.columns)}")
        print(f"   • Columns: {list(df_result.columns)}")
        
        # Show data types
        print(f"\n📋 Data Summary:")
        display(df_result.info())
        
        # Show successful vs failed scraping
        error_mask = df_result['coin'].str.contains('error', na=False) | df_result.isnull().any(axis=1)
        successful_scrapes = len(df_result[~error_mask])
        failed_scrapes = len(df_result[error_mask])
        
        print(f"\n✅ Successful scrapes: {successful_scrapes}")
        print(f"❌ Failed scrapes: {failed_scrapes}")
        
        # Display sample of successful data
        successful_data = df_result[~error_mask]
        if len(successful_data) > 0:
            print(f"\n🎯 Sample of successfully scraped data:")
            display(successful_data)
        
    else:
        print("❌ No data available for analysis. Please run the scraping process first.")
        
except Exception as e:
    print(f"❌ Error in data analysis: {e}")

📊 Data Analysis:

🔢 Dataset Info:
   • Total rows: 4
   • Total columns: 8
   • Columns: ['coin', 'price', 'date', 'Market cap', 'Fully diluted market cap', 'Volume (24h)', '24h volume / market cap', '24h high']

📋 Data Summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   coin                      4 non-null      object
 1   price                     4 non-null      object
 2   date                      4 non-null      object
 3   Market cap                4 non-null      object
 4   Fully diluted market cap  4 non-null      object
 5   Volume (24h)              4 non-null      object
 6   24h volume / market cap   4 non-null      object
 7   24h high                  4 non-null      object
dtypes: object(8)
memory usage: 388.0+ bytes


None


✅ Successful scrapes: 4
❌ Failed scrapes: 0

🎯 Sample of successfully scraped data:


,coin,price,date,Market cap,Fully diluted market cap,Volume (24h),24h volume / market cap,24h high
0,bitcoin,"$105,624.65",Last updated as of 2025-06-04 08:31:52（UTC+0）,"$2,099,205,436,431.22","$2,099,205,436,431.22","$44,509,268,141.79",2.12%,"$106,863.55"
1,ethereum,"$2,637.98",Last updated as of 2025-06-04 08:28:35（UTC+0）,"$318,464,212,981.91","$318,464,212,981.91","$16,440,607,837.49",5.16%,"$2,652.42"
2,binance,$671.57,Last updated as of 2025-06-04 08:29:56（UTC+0）,"$94,616,160,169.44","$94,616,160,169.44","$1,637,858,051.07",1.73%,$671.73
3,solana,$156.85,Last updated as of 2025-06-04 08:28:45（UTC+0）,"$81,961,487,082.61","$81,961,487,082.61","$3,336,282,104.86",4.07%,$163.47


## 10. Troubleshooting & Testing

Use this section for debugging and testing individual components.

In [10]:
# Test single coin scraping (for debugging)
async def test_single_coin(coin_name="bitcoin"):
    """Test scraping a single coin for debugging purposes"""
    print(f"🧪 Testing single coin scraping for: {coin_name}")
    
    try:
        result = await crawl_single_coin(coin_name)
        print(f"\n✅ Test result for {coin_name}:")
        for key, value in result.items():
            print(f"   {key}: {value}")
        return result
    except Exception as e:
        print(f"❌ Test failed for {coin_name}: {e}")
        return None

# Uncomment the line below to test a single coin
# test_result = await test_single_coin("bitcoin")

print("🔧 Testing functions ready. Uncomment the line above to test a single coin.")

🔧 Testing functions ready. Uncomment the line above to test a single coin.


## 📝 Usage Instructions

1. **Setup**: Run cells 1-7 to define all functions and configurations
2. **Execute**: Run cell 8 to start the scraping process
3. **Analyze**: Run cell 9 to analyze the results
4. **Debug**: Use cell 10 for troubleshooting individual coins

## 🔧 Configuration Options

- **list_coin**: Add or remove cryptocurrencies to scrape
- **MAX_RETRIES**: Number of retry attempts for failed requests
- **WAIT_TIME**: Seconds to wait for page loading
- **ELEMENT_TIMEOUT**: Seconds to wait for elements to appear

## 📊 Output Files

- **bitget_coin_data.xlsx**: Excel file with all scraped data
- Console output with real-time progress and statistics

## ⚡ Performance Notes

- Concurrent processing significantly improves speed
- Each coin is scraped in parallel using asyncio tasks
- Retry logic ensures robust data collection
- Error handling prevents single failures from stopping the entire process